# 01 · Project 1: Qwen3-0.6B → strict-JSON entity extractor (LoRA + SFT)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/57Ajay/modeltune/blob/main/notebooks/01_project1_ner_json_lora.ipynb)

Companion to [chapter 09](https://github.com/57Ajay/modeltune/blob/main/course/09_project1_json_extractor.md). The loop you'll repeat for every fine-tune:
**look at data → measure baseline → train → measure again → inspect failures → iterate.**

Config: [`configs/p1_qwen3_ner_json_lora.yaml`](https://github.com/57Ajay/modeltune/blob/main/configs/p1_qwen3_ner_json_lora.yaml). Change `model.id` or the `data` block and the same cells fine-tune a different model on different data.

## 0. Setup
Runtime → Change runtime type → **T4 GPU**. Any newer GPU works too; the code picks fp16 or bf16 by itself.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# Get the course code, then install the pinned libraries from pyproject.toml with uv.
# Colab already ships a CUDA build of PyTorch; `uv pip install --system` keeps it instead of downloading another.
!test -d modeltune || git clone -q https://github.com/57Ajay/modeltune.git
%cd modeltune
!git pull -q
!command -v uv >/dev/null || pip install -q uv
!uv pip install --system -q -r pyproject.toml

In [ ]:
# Hugging Face token: add HF_TOKEN in Colab's key icon (Secrets) panel, with WRITE access to push models.
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    from huggingface_hub import login
    login()
# Pushing to someone else's namespace fails: change hub.repo_id in the configs (or override it) to YOUR username.

## 1. Look at the data the model will see
Raw CoNLL-2003 rows are token lists + BIO tags; the formatter turns them into a system/user prompt and a JSON completion.

In [ ]:
from modeltune.config import load_config
from modeltune.data import load_train_eval
cfg = load_config("configs/p1_qwen3_ner_json_lora.yaml")
train, val = load_train_eval(cfg)
print(train)
ex = train[0]
for m in ex["prompt"] + ex["completion"]:
    print(f"[{m['role']}] {m['content']}\n")

## 2. Baseline: how good is the model *before* fine-tuning?
Never skip this. Note `json_strict` (is the WHOLE reply valid JSON?) vs `f1` (are the entities right?).

In [ ]:
!python -m modeltune.evaluate configs/p1_qwen3_ner_json_lora.yaml --base

## 3. Fine-tune
Read the printed *tokenized training example*: the second block is the only part the loss is computed on (the JSON answer).

In [ ]:
!python -m modeltune.train configs/p1_qwen3_ner_json_lora.yaml

## 4. Evaluate the fine-tuned adapter

In [ ]:
!python -m modeltune.evaluate configs/p1_qwen3_ner_json_lora.yaml

In [ ]:
import json, pandas as pd
rows = {tag: json.load(open(f"outputs/p1-qwen3-0.6b-ner-json-lora/eval_{tag}.json"))["results"] for tag in ("base", "finetuned")}
pd.DataFrame(rows).round(3)

## 5. Look at the failures (the most valuable 10 minutes of any project)

In [ ]:
from modeltune.metrics import parse_json
samples = json.load(open("outputs/p1-qwen3-0.6b-ner-json-lora/eval_finetuned.json"))["samples"]
wrong = [s for s in samples if parse_json(s["prediction"]) != json.loads(s["reference"])]
print(f"{len(wrong)} / {len(samples)} not exactly right\n")
for s in wrong[:8]:
    print("TEXT  :", s["prompt"][-1]["content"]); print("MODEL :", s["prediction"]); print("GOLD  :", s["reference"], "\n")

## 6. Try it on your own text

In [ ]:
!python -m modeltune.chat configs/p1_qwen3_ner_json_lora.yaml --prompt "Sundar Pichai said Google will open a new office in Gurugram before the Diwali holidays." 

## 7. (Optional) Merge the adapter into the base weights and push a standalone model

In [ ]:
!python -m modeltune.merge configs/p1_qwen3_ner_json_lora.yaml --push

## Experiments (keep a results table: config change → f1 / json_strict / minutes)
1. `lora.r=4` vs `lora.r=64`. Does capacity matter for a narrow task?
2. `train.learning_rate=2e-5` (full-FT-style LR) vs `1e-3`. Which fails, and how does the loss curve show it?
3. `data.max_train_samples=500` vs `12000`. How much data do you actually need?
4. `model.id=Qwen/Qwen3-1.7B` (same family, bigger) or `model.id=HuggingFaceTB/SmolLM2-360M-Instruct` (different family). Nothing else changes; that's the point of the config design.
5. `lora.target_modules=[q_proj,v_proj]` (the original LoRA paper's choice) vs `all-linear`.

Pass overrides on the command line, e.g. `!python -m modeltune.train {C} lora.r=4 run_name=p1-r4 hub.push=false`.